In [ ]:
# IMDB Sentiment Analysis using SimpleRNN

# 1. Import Libraries
# 2. Load IMDB Dataset
# 3. Explore Dataset
# 4. Word Index
# 5. Decode Reviews
# 6. Padding
# 7. Build Model
# 8. Model Summary
# 9. Compile Model
# 10. Train Model
# 11. Evaluate Model
# 12. Predict Reviews
# 13. Save Model

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [ ]:
vocab_size=10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words = vocab_size
)
print("Training Reviews: ", len(X_train))
print("Testing Reviews: ", len(X_test))

print("\n First Review(Encoded): ")
print(X_train[0])

print("\nFirst Label:")
print(y_train[0])

#word index
word_index = imdb.get_word_index()

reverse_word_index = {
    value+3: key
    for key, value in word_index.items()
}

reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

##Decode Review

def decode_review(encoded_review):
    return " ".join(
        reverse_word_index.get(i, "?")
        for i in encoded_review
    )

print("\n Decoded Review:\n")
print(decode_review(X_train[0]))

#padding

max_length = 200

X_train = pad_sequences(
    X_train, 
    maxlen = max_length,
    padding ="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test, 
    maxlen = max_length,
    padding="post",
    truncating="post"
)

print("\n shape After Padding")

print(X_train.shape)

##build model 

model = Sequential()

model.add(
    Embedding(
        input_dim = vocab_size,
        output_dim=128,
        input_length = max_length
    )
)

model.add(
    SimpleRNN(
        units=64,
        activation="tanh"
    )
)

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)


##model summary 

model.summary()

##compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

##train model

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

##evaluate model

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\n Test loss: ", loss)
print("Test accuracy: ", accuracy)

##prediction

prediction = model.predict(X_test[:5])
print("\n prediction probabilities")
print(prediction)

print("\n predicted labels")
print((prediction > 0.5).astype(int))

print("\n Actual labels")
print(y_test[:5])

# model.save("imdb_rnn_model.keras")
# print("\n model saved successfully")